In [1]:
%run ../svm/forest_classification_best.ipynb

Forest points: 250
Non-forest points: 250
[[71, 0], [0, 81]]
Accuracy: 1
Kappa: 1


In [2]:
adm = ee.FeatureCollection('WM/geoLab/geoBoundaries/600/ADM2')

jabalpur = adm.filter(ee.Filter.eq('shapeGroup', 'IND')).filter(ee.Filter.Or(ee.Filter.eq('shapeName', 'Jabalpur'),ee.Filter.stringContains('shapeName', 'Jabalpur')))

print(jabalpur.size())

ee.Number({
  "functionInvocationValue": {
    "functionName": "Collection.size",
    "arguments": {
      "collection": {
        "functionInvocationValue": {
          "functionName": "Collection.filter",
          "arguments": {
            "collection": {
              "functionInvocationValue": {
                "functionName": "Collection.filter",
                "arguments": {
                  "collection": {
                    "functionInvocationValue": {
                      "functionName": "Collection.loadTable",
                      "arguments": {
                        "tableId": {
                          "constantValue": "WM/geoLab/geoBoundaries/600/ADM2"
                        }
                      }
                    }
                  },
                  "filter": {
                    "functionInvocationValue": {
                      "functionName": "Filter.equals",
                      "arguments": {
                        "leftField": {
             

In [3]:
jbpFeature = jabalpur.first()

jbpGeometry = jbpFeature.geometry()

In [4]:
Map.centerObject(jbpGeometry, 10)

Map.addLayer(jbpGeometry, {'color': 'cyan'}, 'JBP Boundary')

Map

Map(center=[23.231871747254015, 79.97790292531754], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
dataset_new = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterDate('2025-11-01', '2025-12-31')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .map(mask_s2_clouds))
new_image = dataset_new.median().clip(jbpGeometry)
ndvi_new = new_image.normalizedDifference(['B8', 'B4']).rename('NDVI')
new_image = new_image.addBands(ndvi_new)


In [6]:
new_classified = new_image.select(bands).classify(classifier)
new_smooth_classification = new_classified.focalMode(1)


In [7]:

Map2 = geemap.Map()
Map2.centerObject(jbpGeometry, 10)
Map2.addLayer(new_image, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'Test Area RGB')
Map2.addLayer(new_smooth_classification, {'min': 0, 'max': 1, 'palette': ['lightgray', 'darkgreen']}, 'Test Area Forest Map')
Map2


Map(center=[23.231871747254015, 79.97790292531754], controls=(WidgetControl(options=['position', 'transparent_…

In [8]:
test_forest_points = ee.FeatureCollection('users/cosypix/jabalpurforestnew')
test_non_forest_points = ee.FeatureCollection('users/cosypix/jabalpurnonforestnew')
test_points = test_forest_points.merge(test_non_forest_points)
def _count(fc):
    try:
        return int(fc.size().getInfo())
    except Exception as e:
        return None
print("Forest points:", _count(test_forest_points))
print("Non-forest points:", _count(test_non_forest_points))


Forest points: 800
Non-forest points: 800


In [9]:
validation_data = new_image.select(bands).sampleRegions(
    collection=test_points, properties=['label'], scale=10)
validation_data = validation_data.filter(ee.Filter.notNull(bands + ['label']))
validated = validation_data.classify(classifier)
confusion_matrix = validated.errorMatrix(actual='label', predicted='classification')
print("Confusion Matrix: ", confusion_matrix.getInfo())
print("Test Area Accuracy: ", confusion_matrix.accuracy().getInfo())
print("Test Area Kappa: ", confusion_matrix.kappa().getInfo())


Confusion Matrix:  [[785, 15], [14, 786]]
Test Area Accuracy:  0.981875
Test Area Kappa:  0.9637500000000001
